In [11]:
import torch
import torch.nn as nn
import numpy as np
import math
import time

# Load frozen embedding table
embedding_table = np.load("tokenizer/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")


Vocab size : 4096
MiniLM dim : 384


In [18]:
import torch.nn.functional as F 

DIM      = 384
N_HEADS  = 8
N_LAYERS = 8
FFN_DIM  = 768
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)

class LayerNorm(nn.Module):
    """
    y = ((x - mean) / sqrt(var + eps)) * gamma + beta
    gamma, beta are learned per-feature scalars — shape (dim,)
    """
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta  = nn.Parameter(torch.zeros(dim))  # shift

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)           # (B, T, 1)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)  # (B, T, 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)    # (B, T, D)
        return self.gamma * x_norm + self.beta               # (B, T, D)

class MultiHeadCausalAttention(nn.Module):
    """
    Projects input into Q, K, V — splits into H heads — computes scaled
    dot-product attention with a causal mask — concatenates heads — projects out.

    Q = x W_q      shape: (B, T, D)
    K = x W_k      shape: (B, T, D)
    V = x W_v      shape: (B, T, D)

    Reshape to (B, H, T, head_dim), then:
        scores = Q @ Kᵀ / sqrt(head_dim)    (B, H, T, T)
        scores = scores + causal_mask        (upper triangle = -inf)
        weights = softmax(scores, dim=-1)    (B, H, T, T)
        out = weights @ V                    (B, H, T, head_dim)

    Concat heads → (B, T, D), project out via W_o
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads       # 48
        self.scale    = self.head_dim ** -0.5

        self.W_q = nn.Linear(dim, dim, bias=False)
        self.W_k = nn.Linear(dim, dim, bias=False)
        self.W_v = nn.Linear(dim, dim, bias=False)
        self.W_o = nn.Linear(dim, dim, bias=False)

        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape

        # --- Project & split into heads ---
        Q = self.W_q(x)  # (B, T, D)
        K = self.W_k(x)  # (B, T, D)
        V = self.W_v(x)  # (B, T, D)

        # Reshape: (B, T, D) → (B, H, T, head_dim)
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # --- Scaled dot-product attention ---
        # (B, H, T, head_dim) @ (B, H, head_dim, T) → (B, H, T, T)
        scores = (Q @ K.transpose(-2, -1)) * self.scale

        # Causal mask: positions can only attend to themselves and earlier tokens
        # Upper triangle (future tokens) set to -inf → softmax drives them to 0
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)  # (B, H, T, T)
        weights = self.attn_drop(weights)

        # (B, H, T, T) @ (B, H, T, head_dim) → (B, H, T, head_dim)
        out = weights @ V

        # --- Concat heads & project ---
        # (B, H, T, head_dim) → (B, T, H*head_dim) = (B, T, D)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)  # (B, T, D)

class FeedForward(nn.Module):
    """
    Two-layer MLP with GELU activation.
    FFN(x) = GELU(x W_1 + b_1) W_2 + b_2

    Expands dim → ffn_dim (wider representation),
    then projects back down ffn_dim → dim.
    """
    def __init__(self, dim, ffn_dim, dropout=0.1):
        super().__init__()
        self.W_1 = nn.Linear(dim, ffn_dim)       # expand
        self.W_2 = nn.Linear(ffn_dim, dim)       # contract
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.W_1(x))   # (B, T, ffn_dim)
        x = self.drop(x)
        x = self.W_2(x)           # (B, T, dim)
        return x

class TransformerBlock(nn.Module):
    """
    Pre-norm residual block (norm_first=True, same as your original config).

    x = x + Attention(LayerNorm(x))   ← self-attention sub-layer
    x = x + FFN(LayerNorm(x))         ← feed-forward sub-layer

    Pre-norm (LN before the sub-layer) stabilises training at depth
    vs post-norm (LN after the residual add).
    """
    def __init__(self, dim, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.norm_1 = LayerNorm(dim)
        self.attn   = MultiHeadCausalAttention(dim, n_heads, dropout)
        self.norm_2 = LayerNorm(dim)
        self.ffn    = FeedForward(dim, ffn_dim, dropout)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm_1(x)))  # (B, T, D)
        x = x + self.drop(self.ffn(self.norm_2(x)))   # (B, T, D)
        return x

class MicroLM(nn.Module):
    """
    Full architecture:
      1. Frozen MiniLM embedding lookup         → (B, T, 384)
      2. Sinusoidal positional encoding added   → (B, T, 384)
      3. N_LAYERS x TransformerBlock            → (B, T, 384)
      4. Final LayerNorm                        → (B, T, 384)
      5. Linear output head → logits            → (B, T, VOCAB_SIZE)
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("embedding_table", embedding_tensor)

        self.blocks = nn.ModuleList([
            TransformerBlock(DIM, N_HEADS, FFN_DIM, dropout=0.1)
            for _ in range(N_LAYERS)
        ])

        self.norm        = LayerNorm(DIM)
        self.output_head = nn.Linear(DIM, VOCAB_SIZE, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape

        x = self.embedding_table[token_ids]                    # (B, T, 384)
        x = x + sinusoidal_encoding(T, DIM, token_ids.device) # (B, T, 384)

        for block in self.blocks:
            x = block(x)   # (B, T, 384)

        x = self.norm(x)
        return self.output_head(x)  # (B, T, VOCAB_SIZE)

In [20]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")

# print(f"\nParam breakdown:")
# for name, p in model.named_parameters():
#     if p.requires_grad:
#         print(f"  {name:55s} {p.numel():>10,}")

# Forward pass
x      = torch.randint(0, VOCAB_SIZE, (2, 32))
logits = model(x)
print(f"\nInput  : {x.shape}")
print(f"Output : {logits.shape}")

Trainable params : 11,032,320
Frozen params    : 0  (embedding table)
Total params     : 11,032,320

Input  : torch.Size([2, 32])
Output : torch.Size([2, 32, 4096])


In [21]:
# ── Training Config ───────────────────────────────────────────────────────────
import gc
import json
import random
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

SHARD_DIR  = Path("dataset_shards")
SEQ_LEN    = 512
BATCH_SIZE = 32
N_EPOCHS   = 3
LOG_EVERY  = 200
CKPT_DIR   = Path("checkpoints_v1")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Device      : cuda
Model params: 11,032,320


In [22]:
from torch.utils.data import Dataset

SEQ_LEN = 512    # window size
STRIDE  = 448    # step size → 64-token overlap

class ShardDataset(Dataset):
    def __init__(self, shard_path, seq_len=SEQ_LEN, stride=STRIDE):
        flat = torch.load(shard_path, weights_only=True)

        self.flat    = flat.to(torch.int16)
        self.seq_len = seq_len
        self.stride  = stride

        # number of full windows we can extract
        # +1 because we need seq_len+1 tokens (x + y target)
        n_tokens = len(self.flat)
        self.num_samples = max(0, (n_tokens - seq_len - 1) // stride + 1)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start  = idx * self.stride
        tokens = self.flat[start : start + self.seq_len + 1].to(torch.long)  # ← cast here
        
        if len(tokens) < self.seq_len + 1:
            pad    = torch.full((self.seq_len + 1 - len(tokens),), PAD_ID, dtype=torch.long)
            tokens = torch.cat([tokens, pad])

        x = tokens[:-1]
        y = tokens[1:]
        return x, y


BOS_ID       = 2
EOS_ID       = 3

def collate_fn(batch):
    xs, ys = zip(*batch)
    # all samples are now fixed-length (seq_len), so no padding needed here
    x_pad = torch.stack(xs)          # [B, seq_len]
    y_pad = torch.stack(ys).clone()  # [B, seq_len]

    # mask out targets after the first EOS in each sequence
    for i, y in enumerate(y_pad):
        eos_positions = (y == EOS_ID).nonzero(as_tuple=True)[0]
        if len(eos_positions) > 0:
            eos_pos = eos_positions[0].item()
            if eos_pos + 1 < y.size(0):
                y_pad[i, eos_pos + 1:] = -100

    return x_pad, y_pad

In [6]:
from tokenizers import Tokenizer
TOKENIZER_PATH   = "tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
dataset = ShardDataset("dataset_shards/shard_0000.pt")
for i in range(10,11):
    x, y = dataset[i]
    print(f"\nSample {i}")
    print(f"  x: {x.tolist()}")
    print(f"  y: {tokenizer.decode(x.tolist(), skip_special_tokens=False)}")
    # print(f"  y: {y.tolist()}")
for idx  in x.tolist():
    tok = tokenizer.decode([idx], skip_special_tokens=False)
    print(f"{idx} === '{tok}' ")



Sample 10
  x: [20, 378, 455, 350, 539, 20, 181, 2690, 18, 220, 446, 742, 603, 894, 847, 236, 3012, 535, 20, 894, 380, 18, 324, 1320, 892, 1345, 222, 2651, 93, 511, 238, 287, 609, 468, 628, 238, 886, 908, 551, 300, 20, 378, 1278, 236, 1345, 222, 2651, 93, 511, 238, 392, 683, 20, 181, 924, 222, 909, 18, 628, 238, 886, 1296, 308, 3537, 296, 708, 20, 378, 1198, 849, 609, 238, 761, 252, 222, 868, 693, 397, 20, 3, 2, 3515, 238, 628, 455, 2916, 89, 20, 378, 949, 236, 392, 304, 1192, 238, 2285, 20, 735, 391, 587, 350, 584, 236, 542, 1487, 238, 3447, 304, 430, 347, 89, 20, 378, 908, 430, 347, 89, 455, 250, 884, 238, 1193, 20, 181, 520, 397, 18, 1487, 238, 3447, 427, 236, 506, 469, 294, 220, 1037, 20, 378, 1206, 927, 238, 628, 236, 287, 708, 238, 387, 611, 430, 347, 89, 20, 378, 380, 391, 568, 2047, 646, 220, 3012, 294, 535, 700, 391, 587, 20, 927, 238, 628, 3038, 18, 429, 391, 587, 350, 1217, 1970, 20, 181, 1271, 1847, 326, 1487, 238, 3447, 1633, 18, 927, 238, 628, 1068, 236, 457, 1096, 20, 3

In [23]:
# run once after sharding
# meta = {
#     "total_samples": sum(len(ShardDataset(p)) for p in sorted(SHARD_DIR.glob("shard_*.pt"))),
#     "seq_len": SEQ_LEN,
#     "stride": STRIDE,
# }
# with open(SHARD_DIR / "meta.json", "w") as f:
#     json.dump(meta, f, indent=2)
    
# ── Optimizer & Scheduler ─────────────────────────────────────────────────────

with open(SHARD_DIR / "meta.json") as f:
    meta = json.load(f)

total_steps = (meta['total_samples'] // BATCH_SIZE) * N_EPOCHS

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-5
)
scaler    = torch.cuda.amp.GradScaler()

print(f"Total samples: {meta['total_samples']:,}")
print(f"Total steps  : {total_steps:,}")

Total samples: 5,751,488
Total steps  : 539,202


/tmp/ipykernel_48566/892036717.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


In [8]:
model.to(device)

MicroLM(
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm_1): LayerNorm()
      (attn): MultiHeadCausalAttention(
        (W_q): Linear(in_features=384, out_features=384, bias=False)
        (W_k): Linear(in_features=384, out_features=384, bias=False)
        (W_v): Linear(in_features=384, out_features=384, bias=False)
        (W_o): Linear(in_features=384, out_features=384, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
      )
      (norm_2): LayerNorm()
      (ffn): FeedForward(
        (W_1): Linear(in_features=384, out_features=384, bias=True)
        (W_2): Linear(in_features=384, out_features=384, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm()
  (output_head): Linear(in_features=384, out_features=4096, bias=False)
)

In [25]:
shard_files  = sorted(SHARD_DIR.glob("shard_*.pt"))[:1]
shard_files

[PosixPath('dataset_shards/shard_0000.pt')]

In [26]:
# ── Training Loop with Resume ─────────────────────────────────────────────────
import math

shard_files  = sorted(SHARD_DIR.glob("shard_*.pt"))[:1]
loss_history = []
start_epoch  = 0
start_shard  = 0

# ── Resume from latest checkpoint ────────────────────────────────────────────
# Check mid-epoch checkpoints first, then epoch checkpoints
all_ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
mid_ckpts  = [c for c in all_ckpts if "shard" in c.name]
epoch_ckpts = [c for c in all_ckpts if "shard" not in c.name]

if mid_ckpts:
    latest = mid_ckpts[-1]
    print(f"Resuming from mid-epoch checkpoint {latest}...")
    ckpt = torch.load(latest, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    loss_history = ckpt.get("loss_history", [])
    start_epoch  = ckpt["epoch"] - 1        # epoch is 1-indexed, range is 0-indexed
    start_shard  = ckpt["shard"] + 1        # resume from next shard
    print(f"Resumed — epoch {start_epoch+1}, starting from shard {start_shard+1}")

elif epoch_ckpts:
    latest = epoch_ckpts[-1]
    print(f"Resuming from epoch checkpoint {latest}...")
    ckpt = torch.load(latest, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    loss_history = ckpt.get("loss_history", [])
    start_epoch  = ckpt["epoch"]            # full epoch completed
    start_shard  = 0                        # start from beginning of next epoch
    print(f"Resumed — starting from epoch {start_epoch+1}")

else:
    print("No checkpoint found — training from scratch")

model.to(device)
model.train()

for epoch in range(start_epoch, N_EPOCHS):
    epoch_loss  = 0.0
    epoch_steps = 0

    # no shuffle — sequential shard order
    shard_order = list(range(len(shard_files)))

    # if resuming mid-epoch, skip already completed shards
    if epoch == start_epoch and start_shard > 0:
        shard_order = shard_order[start_shard:]
        print(f"Skipping shards 0-{start_shard-1}, resuming from shard {start_shard}")

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print(f"{'='*60}")

    for shard_num, shard_idx in enumerate(shard_order):
        # just before the step loop:
        shard_start = time.time()
        interval_start = time.time()
        
        # actual shard number in epoch (accounts for skipped shards on resume)
        actual_shard_num = shard_idx  

        shard_path = shard_files[shard_idx]
        print(f"\n[Epoch {epoch+1}] Shard {actual_shard_num+1}/{len(shard_files)} — {shard_path.name}")

        ds     = ShardDataset(shard_path)
        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=0,
            pin_memory=True,
        )

        shard_loss  = 0.0
        shard_steps = 0

        for step, (x, y) in enumerate(loader):
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(x)
                loss   = F.cross_entropy(
                    logits.view(-1, logits.size(-1)),
                    y.view(-1),
                    ignore_index=-100,
                )
                del logits  # free 268MB immediately before backward


            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            if not math.isnan(loss.item()):
                shard_loss  += loss.item()
                shard_steps += 1
                epoch_loss  += loss.item()
                epoch_steps += 1

            if step % LOG_EVERY == 0:
                avg = shard_loss / max(shard_steps, 1)
                ppl = math.exp(min(avg, 20))
                lr  = scheduler.get_last_lr()[0]
                elapsed_total = time.time() - shard_start
                interval_time = time.time() - interval_start
                print(f"  step {step:>5} | loss {loss.item():.4f} | avg {avg:.4f} | ppl {ppl:.2f} | lr {lr:.2e} | last {LOG_EVERY} steps: {interval_time:.1f}s | total: {elapsed_total:.0f}s")
                interval_start = time.time()  # reset for next interval

            del loss, x, y
        shard_avg = shard_loss / max(shard_steps, 1)
        shard_ppl = math.exp(min(shard_avg, 20))
        loss_history.append({
            "epoch":      epoch + 1,
            "shard":      actual_shard_num,
            "avg_loss":   shard_avg,
            "perplexity": shard_ppl,
        })
        print(f"  Shard done — avg loss: {shard_avg:.4f} | ppl {shard_ppl:.2f}")

        del ds, loader
        gc.collect()
        torch.cuda.empty_cache()

        # mid-epoch checkpoint every 5 shards
        if (actual_shard_num + 1) % 2 == 0:
            ckpt_path = CKPT_DIR / f"epoch_{epoch+1}_shard_{actual_shard_num}.pt"
            torch.save({
                "epoch":        epoch + 1,
                "shard":        actual_shard_num,
                "model":        model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "scheduler":    scheduler.state_dict(),
                "loss_history": loss_history,
            }, ckpt_path)
            print(f"Mid-epoch checkpoint saved → {ckpt_path}")

    # reset start_shard after first resumed epoch
    start_shard = 0

    epoch_avg = epoch_loss / max(epoch_steps, 1)
    epoch_ppl = math.exp(min(epoch_avg, 20))
    print(f"\nEpoch {epoch+1} complete — avg loss: {epoch_avg:.4f} | ppl {epoch_ppl:.2f}")

    ckpt_path = CKPT_DIR / f"epoch_{epoch+1}.pt"
    torch.save({
        "epoch":        epoch + 1,
        "shard":        -1,
        "model":        model.state_dict(),
        "optimizer":    optimizer.state_dict(),
        "scheduler":    scheduler.state_dict(),
        "loss_history": loss_history,
    }, ckpt_path)
    print(f"Epoch checkpoint saved → {ckpt_path}")

    with open(CKPT_DIR / "loss_history.json", "w") as f:
        json.dump(loss_history, f, indent=2)

print("\nTraining complete.")

No checkpoint found — training from scratch

Epoch 1/3

[Epoch 1] Shard 1/1 — shard_0000.pt
  step     0 | loss 8.6304 | avg 8.6304 | ppl 5599.48 | lr 3.00e-04 | last 200 steps: 2.4s | total: 2s
  step   200 | loss 5.5521 | avg 6.0289 | ppl 415.26 | lr 3.00e-04 | last 200 steps: 43.0s | total: 45s
  step   400 | loss 5.3388 | avg 5.7510 | ppl 314.51 | lr 3.00e-04 | last 200 steps: 42.1s | total: 88s
  step   600 | loss 4.7236 | avg 5.5159 | ppl 248.61 | lr 3.00e-04 | last 200 steps: 42.3s | total: 130s
  step   800 | loss 4.3472 | avg 5.2884 | ppl 198.03 | lr 3.00e-04 | last 200 steps: 42.3s | total: 172s
  step  1000 | loss 4.2519 | avg 5.0888 | ppl 162.20 | lr 3.00e-04 | last 200 steps: 42.2s | total: 214s
  step  1200 | loss 4.0624 | avg 4.9185 | ppl 136.80 | lr 3.00e-04 | last 200 steps: 42.3s | total: 257s
  step  1400 | loss 3.8862 | avg 4.7675 | ppl 117.62 | lr 3.00e-04 | last 200 steps: 42.2s | total: 299s
  step  1600 | loss 3.7193 | avg 4.6339 | ppl 102.91 | lr 3.00e-04 | las